In [4]:
pip install timm -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
import torchvision.transforms as T
from torch.utils.data import Dataset
    

from PIL import Image
import matplotlib.pyplot as plt

import os
import random
import numpy as np
import cv2

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

import time
from datetime import datetime

from zoneinfo import ZoneInfo

## Resize & Split Dataset

In [6]:
import os
import glob
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
import torchvision.transforms as T
from tqdm.auto import tqdm
import random

# --- KONFIGURASI ---
INPUT_DIR = '/kaggle/input/anime-faces-waifu2x'
OUTPUT_DIR = '/kaggle/working/dataset_split'
EXTENSIONS = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG'] 

# Ukuran Target (FIXED)
TARGET_HR_SIZE = (128, 128)  # Target Ground Truth
TARGET_LR_SIZE = (64, 64)    # Target Input

# Rasio Split (Hanya Train dan Test)
TRAIN_RATIO = 0.9
TEST_RATIO = 0.1  # Pastikan totalnya 1.0 (atau mendekati)

LIMIT_DATA = 10 # Ubah jadi None jika ingin semua data

# --- FUNGSI UTAMA ---

def create_dir_structure(base_path):
    # HANYA TRAIN DAN TEST
    splits = ['train', 'test']
    types = ['HR', 'LR']
    
    if os.path.exists(base_path):
        shutil.rmtree(base_path)
        
    for split in splits:
        for dtype in types:
            os.makedirs(os.path.join(base_path, split, dtype), exist_ok=True)
    print(f"Folder output dibuat (bersih) di: {base_path}")

def get_image_paths(directory):
    all_files = []
    for ext in EXTENSIONS:
        all_files.extend(glob.glob(os.path.join(directory, '**', ext), recursive=True))
    return all_files

def process_and_save(file_paths, split_name, output_base_dir):
    hr_dest_dir = os.path.join(output_base_dir, split_name, 'HR')
    lr_dest_dir = os.path.join(output_base_dir, split_name, 'LR')
    
    if not file_paths:
        return

    print(f"Memproses {split_name} data ({len(file_paths)} gambar)...")
    
    # Definisi Transformasi
    transform_hr = T.Compose([
        T.Resize(TARGET_HR_SIZE, interpolation=T.InterpolationMode.BICUBIC)
    ])
    
    transform_lr = T.Compose([
        T.Resize(TARGET_LR_SIZE, interpolation=T.InterpolationMode.BICUBIC)
    ])
    
    for file_path in tqdm(file_paths):
        try:
            filename = os.path.basename(file_path)
            file_stem = os.path.splitext(filename)[0]
            save_name = f"{file_stem}.png" 
            
            # Buka Gambar Asli
            img_source = Image.open(file_path).convert('RGB')
            
            # --- LANGKAH 1: Buat HR (128x128) ---
            img_hr = transform_hr(img_source)
            img_hr.save(os.path.join(hr_dest_dir, save_name))
            
            # --- LANGKAH 2: Buat LR (64x64) ---
            img_lr = transform_lr(img_hr)
            img_lr.save(os.path.join(lr_dest_dir, save_name))
            
        except Exception as e:
            print(f"Gagal memproses {file_path}: {e}")

# --- EKSEKUSI ---

print("Mencari file gambar...")
all_images = get_image_paths(INPUT_DIR)
print(f"Ditemukan total {len(all_images)} gambar.")

if len(all_images) == 0:
    print("Tidak ada gambar ditemukan! Periksa path INPUT_DIR.")
else:
    # Random Seed
    random.seed(42) 
    random.shuffle(all_images)
    
    # Limit Data Check
    if LIMIT_DATA is not None and isinstance(LIMIT_DATA, int):
        if LIMIT_DATA < len(all_images):
            print('Mode Limit aktif!!! Limit data:', LIMIT_DATA)
            all_images = all_images[:LIMIT_DATA]
        else:
            print('Limit data lebih besar dari total data, mengambil semua data.')
    
    if len(all_images) < 2:
        print("PERINGATAN: Data terlalu sedikit untuk di-split.")

    # --- SPLIT LOGIC BARU (Hanya Train & Test) ---
    train_imgs, test_imgs = train_test_split(
        all_images, 
        train_size=TRAIN_RATIO, 
        random_state=42,
        shuffle=True
    )
    
    print(f"Jumlah Train: {len(train_imgs)}")
    print(f"Jumlah Test: {len(test_imgs)}")

    create_dir_structure(OUTPUT_DIR)

    # Proses simpan gambar
    process_and_save(train_imgs, 'train', OUTPUT_DIR)
    process_and_save(test_imgs, 'test', OUTPUT_DIR)

    print(f"\nSelesai! HR (128x128) dan LR (64x64) tersimpan di folder: {OUTPUT_DIR}")

Mencari file gambar...
Ditemukan total 0 gambar.
Tidak ada gambar ditemukan! Periksa path INPUT_DIR.


In [7]:
# -----------------------------------------------------------------------------------
# SwinIR: Image Restoration Using Swin Transformer, https://arxiv.org/abs/2108.10257
# Originally Written by Ze Liu, Modified by Jingyun Liang.
# -----------------------------------------------------------------------------------

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


def window_partition(x, window_size):
    """
    Args:
        x: (B, H, W, C)
        window_size (int): window size

    Returns:
        windows: (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    Args:
        windows: (num_windows*B, window_size, window_size, C)
        window_size (int): Window size
        H (int): Height of image
        W (int): Width of image

    Returns:
        x: (B, H, W, C)
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    r""" Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.

    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): The height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set
        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0
        proj_drop (float, optional): Dropout ratio of output. Default: 0.0
    """

    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))  # 2*Wh-1 * 2*Ww-1, nH

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)

        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        """
        Args:
            x: input features with shape of (num_windows*B, N, C)
            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None
        """
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            attn = self.softmax(attn)
        else:
            attn = self.softmax(attn)

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    def extra_repr(self) -> str:
        return f'dim={self.dim}, window_size={self.window_size}, num_heads={self.num_heads}'

    def flops(self, N):
        # calculate flops for 1 window with token length of N
        flops = 0
        # qkv = self.qkv(x)
        flops += N * self.dim * 3 * self.dim
        # attn = (q @ k.transpose(-2, -1))
        flops += self.num_heads * N * (self.dim // self.num_heads) * N
        #  x = (attn @ v)
        flops += self.num_heads * N * N * (self.dim // self.num_heads)
        # x = self.proj(x)
        flops += N * self.dim * self.dim
        return flops


class SwinTransformerBlock(nn.Module):
    r""" Swin Transformer Block.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resulotion.
        num_heads (int): Number of attention heads.
        window_size (int): Window size.
        shift_size (int): Shift size for SW-MSA.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float, optional): Stochastic depth rate. Default: 0.0
        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if self.shift_size > 0:
            attn_mask = self.calculate_mask(self.input_resolution)
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)

    def calculate_mask(self, x_size):
        # calculate attention mask for SW-MSA
        H, W = x_size
        img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
        h_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))

        return attn_mask

    def forward(self, x, x_size):
        H, W = x_size
        B, L, C = x.shape
        # assert L == H * W, "input feature has wrong size"

        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        # partition windows
        x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA (to be compatible for testing on images whose shapes are the multiple of window size
        if self.input_resolution == x_size:
            attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C
        else:
            attn_windows = self.attn(x_windows, mask=self.calculate_mask(x_size).to(x.device))

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C

        # reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        x = x.view(B, H * W, C)

        # FFN
        x = shortcut + self.drop_path(x)
        x = x + self.drop_path(self.mlp(self.norm2(x)))

        return x

    def extra_repr(self) -> str:
        return f"dim={self.dim}, input_resolution={self.input_resolution}, num_heads={self.num_heads}, " \
               f"window_size={self.window_size}, shift_size={self.shift_size}, mlp_ratio={self.mlp_ratio}"

    def flops(self):
        flops = 0
        H, W = self.input_resolution
        # norm1
        flops += self.dim * H * W
        # W-MSA/SW-MSA
        nW = H * W / self.window_size / self.window_size
        flops += nW * self.attn.flops(self.window_size * self.window_size)
        # mlp
        flops += 2 * H * W * self.dim * self.dim * self.mlp_ratio
        # norm2
        flops += self.dim * H * W
        return flops


class PatchMerging(nn.Module):
    r""" Patch Merging Layer.

    Args:
        input_resolution (tuple[int]): Resolution of input feature.
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(self, input_resolution, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x):
        """
        x: B, H*W, C
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."

        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = self.norm(x)
        x = self.reduction(x)

        return x

    def extra_repr(self) -> str:
        return f"input_resolution={self.input_resolution}, dim={self.dim}"

    def flops(self):
        H, W = self.input_resolution
        flops = H * W * self.dim
        flops += (H // 2) * (W // 2) * 4 * self.dim * 2 * self.dim
        return flops


class BasicLayer(nn.Module):
    """ A basic Swin Transformer layer for one stage.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution.
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
    """

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None, use_checkpoint=False):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 drop=drop, attn_drop=attn_drop,
                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                                 norm_layer=norm_layer)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim, norm_layer=norm_layer)
        else:
            self.downsample = None

    def forward(self, x, x_size):
        for blk in self.blocks:
            if self.use_checkpoint:
                x = checkpoint.checkpoint(blk, x, x_size)
            else:
                x = blk(x, x_size)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

    def extra_repr(self) -> str:
        return f"dim={self.dim}, input_resolution={self.input_resolution}, depth={self.depth}"

    def flops(self):
        flops = 0
        for blk in self.blocks:
            flops += blk.flops()
        if self.downsample is not None:
            flops += self.downsample.flops()
        return flops


class RSTB(nn.Module):
    """Residual Swin Transformer Block (RSTB).

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution.
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
        img_size: Input image size.
        patch_size: Patch size.
        resi_connection: The convolutional block before residual connection.
    """

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, downsample=None, use_checkpoint=False,
                 img_size=224, patch_size=4, resi_connection='1conv'):
        super(RSTB, self).__init__()

        self.dim = dim
        self.input_resolution = input_resolution

        self.residual_group = BasicLayer(dim=dim,
                                         input_resolution=input_resolution,
                                         depth=depth,
                                         num_heads=num_heads,
                                         window_size=window_size,
                                         mlp_ratio=mlp_ratio,
                                         qkv_bias=qkv_bias, qk_scale=qk_scale,
                                         drop=drop, attn_drop=attn_drop,
                                         drop_path=drop_path,
                                         norm_layer=norm_layer,
                                         downsample=downsample,
                                         use_checkpoint=use_checkpoint)

        if resi_connection == '1conv':
            self.conv = nn.Conv2d(dim, dim, 3, 1, 1)
        elif resi_connection == '3conv':
            # to save parameters and memory
            self.conv = nn.Sequential(nn.Conv2d(dim, dim // 4, 3, 1, 1), nn.LeakyReLU(negative_slope=0.2, inplace=True),
                                      nn.Conv2d(dim // 4, dim // 4, 1, 1, 0),
                                      nn.LeakyReLU(negative_slope=0.2, inplace=True),
                                      nn.Conv2d(dim // 4, dim, 3, 1, 1))

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=0, embed_dim=dim,
            norm_layer=None)

        self.patch_unembed = PatchUnEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=0, embed_dim=dim,
            norm_layer=None)

    def forward(self, x, x_size):
        return self.patch_embed(self.conv(self.patch_unembed(self.residual_group(x, x_size), x_size))) + x

    def flops(self):
        flops = 0
        flops += self.residual_group.flops()
        H, W = self.input_resolution
        flops += H * W * self.dim * self.dim * 9
        flops += self.patch_embed.flops()
        flops += self.patch_unembed.flops()

        return flops


class PatchEmbed(nn.Module):
    r""" Image to Patch Embedding

    Args:
        img_size (int): Image size.  Default: 224.
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.num_patches = patches_resolution[0] * patches_resolution[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        if norm_layer is not None:
            self.norm = norm_layer(embed_dim)
        else:
            self.norm = None

    def forward(self, x):
        x = x.flatten(2).transpose(1, 2)  # B Ph*Pw C
        if self.norm is not None:
            x = self.norm(x)
        return x

    def flops(self):
        flops = 0
        H, W = self.img_size
        if self.norm is not None:
            flops += H * W * self.embed_dim
        return flops


class PatchUnEmbed(nn.Module):
    r""" Image to Patch Unembedding

    Args:
        img_size (int): Image size.  Default: 224.
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.num_patches = patches_resolution[0] * patches_resolution[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

    def forward(self, x, x_size):
        B, HW, C = x.shape
        x = x.transpose(1, 2).view(B, self.embed_dim, x_size[0], x_size[1])  # B Ph*Pw C
        return x

    def flops(self):
        flops = 0
        return flops


class Upsample(nn.Sequential):
    """Upsample module.

    Args:
        scale (int): Scale factor. Supported scales: 2^n and 3.
        num_feat (int): Channel number of intermediate features.
    """

    def __init__(self, scale, num_feat):
        m = []
        if (scale & (scale - 1)) == 0:  # scale = 2^n
            for _ in range(int(math.log(scale, 2))):
                m.append(nn.Conv2d(num_feat, 4 * num_feat, 3, 1, 1))
                m.append(nn.PixelShuffle(2))
        elif scale == 3:
            m.append(nn.Conv2d(num_feat, 9 * num_feat, 3, 1, 1))
            m.append(nn.PixelShuffle(3))
        else:
            raise ValueError(f'scale {scale} is not supported. ' 'Supported scales: 2^n and 3.')
        super(Upsample, self).__init__(*m)


class UpsampleOneStep(nn.Sequential):
    """UpsampleOneStep module (the difference with Upsample is that it always only has 1conv + 1pixelshuffle)
       Used in lightweight SR to save parameters.

    Args:
        scale (int): Scale factor. Supported scales: 2^n and 3.
        num_feat (int): Channel number of intermediate features.

    """

    def __init__(self, scale, num_feat, num_out_ch, input_resolution=None):
        self.num_feat = num_feat
        self.input_resolution = input_resolution
        m = []
        m.append(nn.Conv2d(num_feat, (scale ** 2) * num_out_ch, 3, 1, 1))
        m.append(nn.PixelShuffle(scale))
        super(UpsampleOneStep, self).__init__(*m)

    def flops(self):
        H, W = self.input_resolution
        flops = H * W * self.num_feat * 3 * 9
        return flops


class SwinIR(nn.Module):
    r""" SwinIR
        A PyTorch impl of : `SwinIR: Image Restoration Using Swin Transformer`, based on Swin Transformer.

    Args:
        img_size (int | tuple(int)): Input image size. Default 64
        patch_size (int | tuple(int)): Patch size. Default: 1
        in_chans (int): Number of input image channels. Default: 3
        embed_dim (int): Patch embedding dimension. Default: 96
        depths (tuple(int)): Depth of each Swin Transformer layer.
        num_heads (tuple(int)): Number of attention heads in different layers.
        window_size (int): Window size. Default: 7
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4
        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float): Override default qk scale of head_dim ** -0.5 if set. Default: None
        drop_rate (float): Dropout rate. Default: 0
        attn_drop_rate (float): Attention dropout rate. Default: 0
        drop_path_rate (float): Stochastic depth rate. Default: 0.1
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False
        patch_norm (bool): If True, add normalization after patch embedding. Default: True
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False
        upscale: Upscale factor. 2/3/4/8 for image SR, 1 for denoising and compress artifact reduction
        img_range: Image range. 1. or 255.
        upsampler: The reconstruction reconstruction module. 'pixelshuffle'/'pixelshuffledirect'/'nearest+conv'/None
        resi_connection: The convolutional block before residual connection. '1conv'/'3conv'
    """

    def __init__(self, img_size=64, patch_size=1, in_chans=3,
                 embed_dim=96, depths=[6, 6, 6, 6], num_heads=[6, 6, 6, 6],
                 window_size=7, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 norm_layer=nn.LayerNorm, ape=False, patch_norm=True,
                 use_checkpoint=False, upscale=2, img_range=1., upsampler='', resi_connection='1conv',
                 **kwargs):
        super(SwinIR, self).__init__()
        num_in_ch = in_chans
        num_out_ch = in_chans
        num_feat = 64
        self.img_range = img_range
        if in_chans == 3:
            rgb_mean = (0.4488, 0.4371, 0.4040)
            self.mean = torch.Tensor(rgb_mean).view(1, 3, 1, 1)
        else:
            self.mean = torch.zeros(1, 1, 1, 1)
        self.upscale = upscale
        self.upsampler = upsampler
        self.window_size = window_size

        #####################################################################################################
        ################################### 1, shallow feature extraction ###################################
        self.conv_first = nn.Conv2d(num_in_ch, embed_dim, 3, 1, 1)

        #####################################################################################################
        ################################### 2, deep feature extraction ######################################
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.num_features = embed_dim
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=embed_dim, embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)
        num_patches = self.patch_embed.num_patches
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # merge non-overlapping patches into image
        self.patch_unembed = PatchUnEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=embed_dim, embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)

        # absolute position embedding
        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule

        # build Residual Swin Transformer blocks (RSTB)
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = RSTB(dim=embed_dim,
                         input_resolution=(patches_resolution[0],
                                           patches_resolution[1]),
                         depth=depths[i_layer],
                         num_heads=num_heads[i_layer],
                         window_size=window_size,
                         mlp_ratio=self.mlp_ratio,
                         qkv_bias=qkv_bias, qk_scale=qk_scale,
                         drop=drop_rate, attn_drop=attn_drop_rate,
                         drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],  # no impact on SR results
                         norm_layer=norm_layer,
                         downsample=None,
                         use_checkpoint=use_checkpoint,
                         img_size=img_size,
                         patch_size=patch_size,
                         resi_connection=resi_connection

                         )
            self.layers.append(layer)
        self.norm = norm_layer(self.num_features)

        # build the last conv layer in deep feature extraction
        if resi_connection == '1conv':
            self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, 1, 1)
        elif resi_connection == '3conv':
            # to save parameters and memory
            self.conv_after_body = nn.Sequential(nn.Conv2d(embed_dim, embed_dim // 4, 3, 1, 1),
                                                 nn.LeakyReLU(negative_slope=0.2, inplace=True),
                                                 nn.Conv2d(embed_dim // 4, embed_dim // 4, 1, 1, 0),
                                                 nn.LeakyReLU(negative_slope=0.2, inplace=True),
                                                 nn.Conv2d(embed_dim // 4, embed_dim, 3, 1, 1))

        #####################################################################################################
        ################################ 3, high quality image reconstruction ################################
        if self.upsampler == 'pixelshuffle':
            # for classical SR
            self.conv_before_upsample = nn.Sequential(nn.Conv2d(embed_dim, num_feat, 3, 1, 1),
                                                      nn.LeakyReLU(inplace=True))
            self.upsample = Upsample(upscale, num_feat)
            self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        elif self.upsampler == 'pixelshuffledirect':
            # for lightweight SR (to save parameters)
            self.upsample = UpsampleOneStep(upscale, embed_dim, num_out_ch,
                                            (patches_resolution[0], patches_resolution[1]))
        elif self.upsampler == 'nearest+conv':
            # for real-world SR (less artifacts)
            self.conv_before_upsample = nn.Sequential(nn.Conv2d(embed_dim, num_feat, 3, 1, 1),
                                                      nn.LeakyReLU(inplace=True))
            self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            if self.upscale == 4:
                self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
            self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
            self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        else:
            # for image denoising and JPEG compression artifact reduction
            self.conv_last = nn.Conv2d(embed_dim, num_out_ch, 3, 1, 1)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'absolute_pos_embed'}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'relative_position_bias_table'}

    def check_image_size(self, x):
        _, _, h, w = x.size()
        mod_pad_h = (self.window_size - h % self.window_size) % self.window_size
        mod_pad_w = (self.window_size - w % self.window_size) % self.window_size
        x = F.pad(x, (0, mod_pad_w, 0, mod_pad_h), 'reflect')
        return x

    def forward_features(self, x):
        x_size = (x.shape[2], x.shape[3])
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        for layer in self.layers:
            x = layer(x, x_size)

        x = self.norm(x)  # B L C
        x = self.patch_unembed(x, x_size)

        return x

    def forward(self, x):
        H, W = x.shape[2:]
        x = self.check_image_size(x)
        
        self.mean = self.mean.type_as(x)
        x = (x - self.mean) * self.img_range

        if self.upsampler == 'pixelshuffle':
            # for classical SR
            x = self.conv_first(x)
            x = self.conv_after_body(self.forward_features(x)) + x
            x = self.conv_before_upsample(x)
            x = self.conv_last(self.upsample(x))
        elif self.upsampler == 'pixelshuffledirect':
            # for lightweight SR
            x = self.conv_first(x)
            x = self.conv_after_body(self.forward_features(x)) + x
            x = self.upsample(x)
        elif self.upsampler == 'nearest+conv':
            # for real-world SR
            x = self.conv_first(x)
            x = self.conv_after_body(self.forward_features(x)) + x
            x = self.conv_before_upsample(x)
            x = self.lrelu(self.conv_up1(torch.nn.functional.interpolate(x, scale_factor=2, mode='nearest')))
            if self.upscale == 4:
                x = self.lrelu(self.conv_up2(torch.nn.functional.interpolate(x, scale_factor=2, mode='nearest')))
            x = self.conv_last(self.lrelu(self.conv_hr(x)))
        else:
            # for image denoising and JPEG compression artifact reduction
            x_first = self.conv_first(x)
            res = self.conv_after_body(self.forward_features(x_first)) + x_first
            x = x + self.conv_last(res)

        x = x / self.img_range + self.mean

        return x[:, :, :H*self.upscale, :W*self.upscale]

    def flops(self):
        flops = 0
        H, W = self.patches_resolution
        flops += H * W * 3 * self.embed_dim * 9
        flops += self.patch_embed.flops()
        for i, layer in enumerate(self.layers):
            flops += layer.flops()
        flops += H * W * 3 * self.embed_dim * self.embed_dim
        flops += self.upsample.flops()
        return flops


if __name__ == '__main__':
    upscale = 4
    window_size = 8
    height = (1024 // upscale // window_size + 1) * window_size
    width = (720 // upscale // window_size + 1) * window_size
    model = SwinIR(upscale=2, img_size=(height, width),
                   window_size=window_size, img_range=1., depths=[6, 6, 6, 6],
                   embed_dim=60, num_heads=[6, 6, 6, 6], mlp_ratio=2, upsampler='pixelshuffledirect')
    print(model)
    print(height, width, model.flops() / 1e9)

    x = torch.randn((1, 3, height, width))
    x = model(x)
    print(x.shape)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


SwinIR(
  (conv_first): Conv2d(3, 60, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (patch_embed): PatchEmbed(
    (norm): LayerNorm((60,), eps=1e-05, elementwise_affine=True)
  )
  (patch_unembed): PatchUnEmbed()
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0): RSTB(
      (residual_group): BasicLayer(
        dim=60, input_resolution=(264, 184), depth=6
        (blocks): ModuleList(
          (0): SwinTransformerBlock(
            dim=60, input_resolution=(264, 184), num_heads=6, window_size=8, shift_size=0, mlp_ratio=2
            (norm1): LayerNorm((60,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              dim=60, window_size=(8, 8), num_heads=6
              (qkv): Linear(in_features=60, out_features=180, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=60, out_features=60, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            

In [8]:
class SobelOperator(nn.Module):
    """ Sobel for detecting edge in image"""
    def __init__(self):
        super(SobelOperator, self).__init__()

        # detecting in horizontal
        sobel_x = torch.tensor([
            [-1,0,1],
            [-2,0,2],
            [-1,0,1]
            ], dtype=torch.float32)

        # detecting in vertical
        sobel_y = torch.tensor([
            [-1,-2,-1],
            [0,0,0],
            [1,2,3]
            ], dtype=torch.float32)

        # reshape convolution [outchannels, in_channel, H,W]
        self.sobel_x = sobel_x.view(1,1,3,3)
        self.sobel_y = sobel_y.view(1,1,3,3)

    def forward(self,x):
        """
        args :
        input tensor : [B,C,H,W]
        returns : gradien magnitude = [B,C,H,W]
        """

        B,C,H,W = x.shape

        # move sobel kernels to same device as input
        sobel_x = self.sobel_x.to(x.device)
        sobel_y = self.sobel_y.to(x.device)

        # expand kernel to all channel
        sobel_x = sobel_x.repeat(C,1,1,1)
        sobel_y = sobel_y.repeat(C,1,1,1)

        
        # Apply sobel operator
        grad_x = F.conv2d(x, sobel_x, padding=1, groups=C)
        grad_y = F.conv2d(x, sobel_y, padding=1, groups=C)

        # compute gradient magnitude
        grad_magnitude = torch.sqrt(grad_x**2 + grad_y ** 2  + 1e-6)

        return grad_magnitude 

In [9]:
class SobelLoss(nn.Module):
    """
    Sobel Loss for image resoration
    measure  different between output and ground gruth
    """

    def __init__(self, loss_weight=1.0, reduction='mean'):
        super(SobelLoss, self).__init__()
        self.sobel = SobelOperator()
        self.loss_weight = loss_weight
        self.reduction = reduction

    def forward(self, pred, target):
        """
        args : 
        pred : predicted image (B,C,H,W)
        target : ground truth image (B,C,H,W)
        returns : sobel loss value
        """
        # extract edges using sobel operator
        pred_edges = self.sobel(pred)
        target_edges = self.sobel(target)

        # compute L1 loss  between edges
        loss = F.l1_loss(pred_edges, target_edges, reduction= self.reduction)

        return self.loss_weight * loss 

class CombinedLoss(nn.Module):
    """
    Combined Loss : pixel loss + Sobel loss
    """
    def __init__(self,
                pixel_weight = 1.0,
                sobel_weight = 0.1,
                use_l1=True,
                use_charbonnier=False):
        super(CombinedLoss, self).__init__()

        self.pixel_weight = pixel_weight
        self.sobel_weight = sobel_weight
        self.use_l1 = use_l1
        self.use_charbonnier = use_charbonnier

        self.sobel_loss = SobelLoss(loss_weight=sobel_weight)

    def forward(self,pred, target):
        """
        args:
        prediced = image predicted [B,C,H,W]
        target = ground truth image [B,C,H,W]
        returns = total loss value"""

        if self.use_l1:
            pixel_loss = F.l1_loss(pred, target)
        elif self.use_charbonnier: 
            eps = 1e-3
            pixel_loss = torch.sqrt((pred-target)**2+eps**2).mean()

        else:
            pixel_loss = F.mse_loss(pred, target)

        # edge loss using sobel
        edge_loss = self.sobel_loss(pred, target)

        # Total loss
        total_loss = self.pixel_weight * pixel_loss + edge_loss

        return total_loss, pixel_loss, edge_loss

class AdaptiveSobelLoss(nn.Module):
    """
    Adaptive Sobel Loss
    the sobel loss are adjusted based on edge strenght
    """
    def __init__(self, 
                pixel_weight=1.0,
                sobel_weight=0.1,
                adaptive=True):
        super(AdaptiveSobelLoss, self).__init__()

        self.pixel_weight = pixel_weight
        self.sobel_weight = sobel_weight
        self.adaptive = adaptive
        self.sobel = SobelOperator()

    def forward(self,pred,target):
        """
        Args:
            pred: predicted image [B, C, H, W]
            target: ground truth image [B, C, H, W]
        Returns:
            total loss value
        """
        pixel_loss = F.l1_loss(pred, target)

        # edge axtraction
        pred_edges = self.sobel(pred)
        target_edges = self.sobel(target)

        if self.adaptive:
            denom = target_edges.max() - target_edges.min()
            edge_weight = (target_edges - target_edges.min()) / (denom + 1e-8)
            edge_loss = (edge_weight * torch.abs(pred_edges-target_edges)).mean()
        else:
            edge_loss = F.l1_loss(pred_edges, target_edges)
    
            # total loss
        total_loss = self.pixel_weight * pixel_loss + self.sobel_weight * edge_loss

        return total_loss, pixel_loss, edge_loss

In [10]:
# ==================== dataset.py ====================
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms


class SRDataset(Dataset):
    """
    Simple dataset untuk Super-Resolution
    Asumsi: folder berisi pasangan LR dan HR dengan nama yang sama
    """
    
    def __init__(self, lr_dir, hr_dir, transform=None):
        """
        Args:
            lr_dir: direktori berisi gambar Low-Resolution
            hr_dir: direktori berisi gambar High-Resolution
            transform: transformasi untuk gambar (optional)
        """
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.transform = transform
        
        # List semua file gambar
        self.image_files = []
        for ext in ['png', 'jpg', 'jpeg', 'bmp']:
            self.image_files.extend([f for f in os.listdir(lr_dir) 
                                    if f.lower().endswith(ext)])
        
        self.image_files.sort()  # Pastikan urutan sama
        print(f'Found {len(self.image_files)} image pairs')
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        
        # Load LR dan HR image
        lr_path = os.path.join(self.lr_dir, img_name)
        hr_path = os.path.join(self.hr_dir, img_name)
        
        lr_img = Image.open(lr_path).convert('RGB')
        hr_img = Image.open(hr_path).convert('RGB')
        
        # Apply transform jika ada
        if self.transform:
            lr_img = self.transform(lr_img)
            hr_img = self.transform(hr_img)
        else:
            # Default: convert ke tensor dan normalize ke [0,1]
            to_tensor = transforms.ToTensor()
            lr_img = to_tensor(lr_img)
            hr_img = to_tensor(hr_img)
        
        return lr_img, hr_img


class SingleFolderDataset(Dataset):
    """
    Dataset untuk folder tunggal (hanya HR images)
    LR akan di-generate on-the-fly dengan downsampling
    """
    
    def __init__(self, hr_dir, scale=4, lr_size=None, hr_size=None):
        """
        Args:
            hr_dir: direktori berisi gambar HR
            scale: faktor downsampling untuk generate LR
            lr_size: ukuran LR yang diinginkan (H, W) - optional
            hr_size: ukuran HR yang diinginkan (H, W) - optional
        """
        self.hr_dir = hr_dir
        self.scale = scale
        self.lr_size = lr_size
        self.hr_size = hr_size
        
        # List semua file gambar
        self.image_files = []
        for ext in ['png', 'jpg', 'jpeg', 'bmp']:
            self.image_files.extend([f for f in os.listdir(hr_dir) 
                                    if f.lower().endswith(ext)])
        
        self.image_files.sort()
        print(f'Found {len(self.image_files)} images')
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        hr_path = os.path.join(self.hr_dir, img_name)
        
        # Load HR image
        hr_img = Image.open(hr_path).convert('RGB')
        
        # Resize HR jika ukuran ditentukan
        if self.hr_size:
            hr_img = transforms.Resize(self.hr_size)(hr_img)
        
        # Generate LR dengan downsampling
        if self.lr_size:
            lr_img = transforms.Resize(self.lr_size, 
                                       interpolation=Image.BICUBIC)(hr_img)
        else:
            # Otomatis berdasarkan scale
            w, h = hr_img.size
            lr_size = (h // self.scale, w // self.scale)
            lr_img = transforms.Resize(lr_size, 
                                       interpolation=Image.BICUBIC)(hr_img)
        
        # Convert ke tensor
        to_tensor = transforms.ToTensor()
        lr_img = to_tensor(lr_img)
        hr_img = to_tensor(hr_img)
        
        return lr_img, hr_img


# ==================== utils.py ====================
import torch
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import cv2


class AverageMeter:
    """Computes and stores the average and current value"""
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def calculate_psnr(img1, img2, max_value=1.0):
    """
    Calculate PSNR between two images
    Args:
        img1, img2: torch tensors [C, H, W] or [B, C, H, W]
        max_value: maximum pixel value (1.0 for normalized)
    """
    if isinstance(img1, torch.Tensor):
        img1 = img1.detach().cpu().numpy()
    if isinstance(img2, torch.Tensor):
        img2 = img2.detach().cpu().numpy()
    
    # Handle batch dimension
    if img1.ndim == 4:
        psnr_list = []
        for i in range(img1.shape[0]):
            psnr_list.append(calculate_psnr(img1[i], img2[i], max_value))
        return np.mean(psnr_list)
    
    # [C, H, W] -> [H, W, C]
    if img1.shape[0] in [1, 3]:
        img1 = np.transpose(img1, (1, 2, 0))
        img2 = np.transpose(img2, (1, 2, 0))
    
    # Squeeze jika grayscale
    if img1.shape[2] == 1:
        img1 = img1.squeeze(2)
        img2 = img2.squeeze(2)
    
    return peak_signal_noise_ratio(img2, img1, data_range=max_value)


def calculate_ssim(img1, img2, max_value=1.0):
    """
    Calculate SSIM between two images
    """
    if isinstance(img1, torch.Tensor):
        img1 = img1.detach().cpu().numpy()
    if isinstance(img2, torch.Tensor):
        img2 = img2.detach().cpu().numpy()
    
    # Handle batch
    if img1.ndim == 4:
        ssim_list = []
        for i in range(img1.shape[0]):
            ssim_list.append(calculate_ssim(img1[i], img2[i], max_value))
        return np.mean(ssim_list)
    
    # [C, H, W] -> [H, W, C]
    if img1.shape[0] in [1, 3]:
        img1 = np.transpose(img1, (1, 2, 0))
        img2 = np.transpose(img2, (1, 2, 0))
    
    channel_axis = 2 if img1.ndim == 3 else None
    
    return structural_similarity(img2, img1, 
                                data_range=max_value,
                                channel_axis=channel_axis)

import cv2
import numpy as np
import torch
import torch.nn.functional as F

def get_edge_mask(img_tensor, threshold=100/255.0):
    """
    Mendeteksi edge dari gambar tensor menggunakan Sobel.
    Args:
        img_tensor: Tensor (B, C, H, W) atau (C, H, W), range [0, 1]
    Returns:
        mask: Tensor binary (1 di tepi, 0 di flat area)
    """
    # Pastikan input adalah tensor 4D (B, C, H, W)
    if img_tensor.dim() == 3:
        img_tensor = img_tensor.unsqueeze(0)
    
    # Konversi ke Grayscale untuk deteksi tepi yang lebih baik (menggunakan bobot RGB standar)
    # R*0.299 + G*0.587 + B*0.114
    gray = 0.299 * img_tensor[:, 0, :, :] + 0.587 * img_tensor[:, 1, :, :] + 0.114 * img_tensor[:, 2, :, :]
    gray = gray.unsqueeze(1) # Kembalikan ke (B, 1, H, W)

    # Definisi Kernel Sobel
    k = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=img_tensor.device)
    k_x = k.view(1, 1, 3, 3)
    k_y = k.t().view(1, 1, 3, 3)

    # Konvolusi untuk mendapatkan gradien
    # Padding=1 agar ukuran output sama dengan input
    gx = F.conv2d(gray, k_x, padding=1)
    gy = F.conv2d(gray, k_y, padding=1)

    # Magnitude gradien
    magnitude = torch.sqrt(gx**2 + gy**2)

    # Buat mask: 1 jika magnitude > threshold, 0 jika tidak
    # Threshold bisa disesuaikan. Untuk anime yang garisnya tegas, nilai 0.3 - 0.5 cukup bagus.
    edge_mask = (magnitude > threshold).float()
    
    return edge_mask

def calculate_edge_psnr(img_sr, img_hr, edge_threshold=0.5):
    """
    Menghitung PSNR hanya pada area tepi (Edge-PSNR).
    Args:
        img_sr: Image Super-Resolution (Tensor), Range [0,1]
        img_hr: Image High-Resolution (Tensor), Range [0,1]
        edge_threshold: Ambang batas deteksi tepi
    """
    # Pastikan input di device yang sama
    device = img_sr.device
    
    # Dapatkan mask tepi dari gambar HR (Ground Truth)
    # Kita menggunakan HR sebagai referensi tepi yang benar
    mask = get_edge_mask(img_hr, threshold=edge_threshold)
    
    # Hitung MSE hanya pada area mask
    # Tambahkan epsilon kecil untuk menghindari pembagian dengan nol jika tidak ada tepi
    mse = (mask * (img_sr - img_hr) ** 2).sum() / (mask.sum() + 1e-8)
    
    # Hitung PSNR
    if mse == 0:
        return float('inf')
    
    edge_psnr = 10 * torch.log10(1.0 / mse)
    
    return edge_psnr.item()


def save_checkpoint(state, filename):
    """Save checkpoint"""
    torch.save(state, filename)
    print(f'Checkpoint saved: {filename}')


def load_checkpoint(model, optimizer, filename):
    """Load checkpoint"""
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint.get('epoch', 0)
    best_psnr = checkpoint.get('best_psnr', 0)
    return epoch, best_psnr


In [11]:
def build_dataloaders(config):
    """ Build dataloader for anime set"""

    train_dataset = SRDataset(
        lr_dir=config.train_lr_dir,
        hr_dir=config.train_hr_dir
    )
    val_dataset = SRDataset(
        lr_dir = config.val_lr_dir,
        hr_dir = config.val_hr_dir
    )

    train_loaders = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers = 0,
        pin_memory=True
    )
    val_loaders = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers = 0,
        pin_memory=True
    )

    return train_loaders, val_loaders


In [12]:
import time
from datetime import datetime


def build_model(config):
    """Build SwinIR model"""
    model = SwinIR(
        upscale=config.scale,
        img_size=(64, 64),
        window_size=config.window_size,
        img_range=1.,
        depths=config.depths,
        embed_dim=config.embed_dim,
        num_heads=config.num_heads,
        mlp_ratio=2,
        upsampler=config.upsampler,
        resi_connection='1conv'
    )
    return model

def build_loss(config):
    # build loss configuration
    if config.loss_type == 'l1':
        print('menggunakan l1')
        return nn.L1Loss()
    elif config.loss_type == 'sobel':
        return SobelLoss(loss_weight=config.sobel_weight)
    elif config.loss_type == 'combined':
        return CombinedLoss(
            pixel_weight = config.pixel_weight,
            sobel_weight = config.sobel_weight,
            use_l1 = True
        )
    elif config.loss_type == 'adaptive':
        return AdaptiveSobelLoss(
            pixel_weight = config.pixel_weight,
            sobel_weight = config.sobel_weight,
            adaptive = True
        )


def train_epoch(model, train_loader, criterion, optimizer, epochs, config):
    """train for one epoch"""
    model.train()

    losses = AverageMeter()
    pixel_losses = AverageMeter()
    edge_losses = AverageMeter()

    pbar = tqdm(train_loader, desc=f'epoch[{epochs}/{config.num_epochs}]')

    for lr_imgs, hr_imgs in pbar:
        lr_images=lr_imgs.to(config.device)
        hr_images=hr_imgs.to(config.device)

        #forward
        sr_imgs = model(lr_images)

        # ensure the same size
        if sr_imgs.shape != hr_images.shape:
            _, _, h,w = hr_images.shape
            sr_imgs = sr_imgs[:,:,:h,:w]

        # compute loss
        if config.loss_type in ['combined', 'adaptive']:
            loss, pixel_loss, edge_loss = criterion(sr_imgs, hr_images)
            pixel_losses.update(pixel_loss.item(), lr_imgs.size(0))
            edge_losses.update(edge_loss.item(), lr_imgs.size(0))
        else:
            loss = criterion(sr_imgs, hr_imgs)
            pixel_loss = loss
            edge_loss = torch.tensor(0.)
        losses.update(loss.item(), lr_imgs.size(0))

        # backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # update progress
        pbar.set_postfix({
            'loss': f'{losses.avg:.4f}',
            'pixel': f'{pixel_losses.avg:.4f}',
            'Edge':f'{edge_losses.avg:.4f}'
        })
    return losses.avg, pixel_losses.avg, edge_losses.avg

@torch.no_grad()
def validate(model, val_loader, epoch, config):
    """Validate Model"""
    model.eval()

    psnr_meter = AverageMeter()
    ssim_meter = AverageMeter()
    edge_psnr_meter = AverageMeter()

    for lr_imgs, hr_imgs in tqdm(val_loader, desc='validating'):
        lr_images = lr_imgs.to(config.device)
        hr_images = hr_imgs.to(config.device)

        #forward
        sr_imgs = model(lr_images)

        # ensure the same size
        if sr_imgs.shape != hr_images.shape:
            _,_,h,w = hr_images.shape
            sr_imgs = sr_imgs[:,:, :h,:w]

        # clamp
        sr_imgs = torch.clamp(sr_imgs,0,1)

        # metrics 
        psnr = calculate_psnr(sr_imgs, hr_images)
        ssim = calculate_ssim(sr_imgs, hr_images)
        edge_psnr = calculate_edge_psnr(sr_imgs, hr_images, edge_threshold=0.15)

        psnr_meter.update(psnr)
        ssim_meter.update(ssim)
        edge_psnr_meter.update(edge_psnr)

        # print(f'\n[Epoch {epoch}] Val - PSNR: {psnr_meter.avg:.4f} dB, SSIM: {ssim_meter.avg:.4f}\n')
    return psnr_meter.avg, ssim_meter.avg, edge_psnr_meter.avg

class AnimeConfig:
    train_lr_dir = '/kaggle/working/dataset_split/train/LR'
    train_hr_dir = '/kaggle/working/dataset_split/train/HR'
    val_lr_dir = '/kaggle/working/dataset_split/test/LR'
    val_hr_dir = '/kaggle/working/dataset_split/test/HR'
    output_dir = '/kaggle/working/experiments_anime'

    # model parameters
    scale = 2 
    window_size = 8
    model_size = 'large'

    if model_size == 'large':
        embed_dim = 180
        depths = [6,6,6,6]
        num_heads = [6,6,6,6]
    else : 
        embed_dim = 60
        depths = [6,6,6,6]
        num_heads = [6,6,6,6]

    upsampler = 'pixelshuffle'

    loss_type = 'l1'
    pixel_weight = 1.0
    sobel_weight = 0.1 # for anime range between 0.1 - 0.5

    # training parameters
    batch_size = 8
    num_epochs = 2
    learning_rate = 2e-4
    betas = (0.9, 0.999)
    eta_min = 1e-7


    # device 
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # experiment name 
    experiment_name = f'anime_waifu2x_{loss_type}_x{scale}'

    if loss_type in ['combined', 'adaptive']:
        experiment_name += f'sobel : {sobel_weight}'


def main():
    config = AnimeConfig()
    
    # Create output directory
    exp_dir = os.path.join(config.output_dir, config.experiment_name)
    os.makedirs(exp_dir, exist_ok=True)
    
    # Print configuration
    print("="*70)
    print("SwinIR Training for Anime Faces Waifu2x")
    print("="*70)
    print(f"Experiment: {config.experiment_name}")
    print(f"Device: {config.device}")
    print(f"Model: {config.model_size} ({config.embed_dim} dim)")
    print(f"Loss: {config.loss_type}", end="")
    if config.loss_type in ['combined', 'adaptive']:
        print(f" (pixel={config.pixel_weight}, sobel={config.sobel_weight})")
    else:
        print()
    print(f"Scale: x{config.scale}")
    print(f"Batch size: {config.batch_size}")
    print(f"Epochs: {config.num_epochs}")
    print(f"Learning rate: {config.learning_rate}")
    print("="*70 + "\n")
    
    # Build model
    print("Building model...")
    model = build_model(config).to(config.device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Model parameters: {num_params / 1e6:.2f}M\n")
    
    # Build loss
    criterion = build_loss(config)
    
    # Build optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        betas=config.betas
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.num_epochs,
        eta_min=config.eta_min
    )
    
    # Build dataloaders
    print("Loading dataset...")
    train_loader, val_loader = build_dataloaders(config)
    print(f"✓ Train samples: {len(train_loader.dataset)}")
    print(f"✓ Val samples: {len(val_loader.dataset)}\n")
    
    # Training loop
    best_psnr = 0.0
    history = {
        'train_loss': [],
        'pixel_loss': [],
        'edge_loss': [],
        'val_psnr': [],
        'val_ssim': [],
        'lr': [],
        'val_edge_psnr': [],
    }
    
    print("="*70)
    print("START TRAINING")
    print("="*70 + "\n")
    
    for epoch in range(1, config.num_epochs + 1):
        epoch_start = time.time()

        # taking lr used for this epoch
        current_lr = optimizer.param_groups[0]['lr']
        # Train
        train_loss, pixel_loss, edge_loss = train_epoch(
            model, train_loader, criterion, optimizer, epoch, config
        )
        scheduler.step()
        # Save history
        history['train_loss'].append(train_loss)
        history['pixel_loss'].append(pixel_loss)
        history['edge_loss'].append(edge_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        # Print summary
        print(f'Epoch {epoch}/{config.num_epochs}:')
        print(f'  Loss: {train_loss:.4f} (Pixel: {pixel_loss:.4f}, Edge: {edge_loss:.4f})')
        print(f'  LR: {current_lr:.6f}')
        
        # testing
        psnr, ssim, edge_psnr = validate(model, val_loader, epoch, config)
        history['val_psnr'].append(psnr)
        history['val_ssim'].append(ssim)
        history['val_edge_psnr'].append(edge_psnr)


        epoch_time = time.time() - epoch_start  # ⏱️ durasi epoch
        # saving model + note
        
        torch.save({
            'epoch': epoch, 
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),

            # hyperparameter
            'learning_rate': current_lr,
            'scale':config.scale,
            'window_size':config.window_size,
            'model_size':config.model_size,
            'loss_type':config.loss_type,
            'pixel_weight':config.pixel_weight,
            'sobel_weight':config.sobel_weight, 
            'betas': config.betas,
            'eta_min': config.eta_min,
            
            # meta tambahan
            "author": "kanza az zahrawani",
            "device": config.device,
            "date": datetime.now(ZoneInfo("Asia/Jakarta")).isoformat(),
            "university": "Universitas Sriwijaya",

            "train_loss":train_loss,
            "pixel_loss": pixel_loss,
            "edge_loss": edge_loss,

            "val_psnr":float(psnr),
            "val_ssim": float(ssim),
            "val_edge_psnr": float(edge_psnr),
            "epoch_time": epoch_time, 
        }, f"model_epoch_{epoch}_{config.loss_type}.pth"
)
        


if __name__ == '__main__':
    main()

SwinIR Training for Anime Faces Waifu2x
Experiment: anime_waifu2x_l1_x2
Device: cuda
Model: large (180 dim)
Loss: l1
Scale: x2
Batch size: 8
Epochs: 2
Learning rate: 0.0002

Building model...
✓ Model parameters: 8.02M

menggunakan l1
Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/dataset_split/train/LR'

In [ ]:
def main_finetuning(
    checkpoint_path, 
    load_optimizer=True,    # Default True untuk resume
    new_lr=None,            # UBAH: Default None agar memakai LR dari checkpoint
    additional_epochs=5,
    new_batch_size=None
):
    # ... (Setup Config & Direktori) ...
    config = AnimeConfig()
    exp_dir = "/kaggle/working/"
    # Override config hanya jika parameter diisi
    if new_lr is not None:
        config.learning_rate = new_lr
    config.num_epochs = additional_epochs
    if new_batch_size: config.batch_size = new_batch_size

    # ... (Build Model & Loss) ...
    model = build_model(config).to(config.device)
    criterion = build_loss(config)

    # 1. Load Checkpoint
    print(f"Loading checkpoint: {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=config.device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Ambil epoch terakhir
    last_epoch = checkpoint.get('epoch', 0)
    start_epoch = last_epoch + 1

    # 2. Setup Optimizer (Inisialisasi awal)
    # Jika new_lr None, kita pakai default config dulu, nanti akan ditimpa oleh load_state_dict
    lr_init = new_lr if new_lr is not None else 1e-4 
    optimizer = optim.Adam(model.parameters(), lr=lr_init, betas=(0.9, 0.999))

    # 3. Load Optimizer State
    if load_optimizer:
        print("Restoring optimizer state...")
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        # LOGIC PENTING:
        # Jika new_lr diisi user -> Kita PAKSA ganti LR (untuk Fine Tuning)
        # Jika new_lr None       -> Kita biarkan LR dari checkpoint (untuk Resume)
        if new_lr is not None:
            print(f"Overriding Learning Rate to: {new_lr}")
            for param_group in optimizer.param_groups:
                param_group['lr'] = new_lr
        else:
            current_lr_check = optimizer.param_groups[0]['lr']
            print(f"Resuming with saved Learning Rate: {current_lr_check:.8f}")

    # 4. Setup Scheduler
    # T_max harus disesuaikan. Jika resume, biasanya kita ingin meneruskan siklusnya,
    # atau membuat siklus baru untuk additional_epochs.
    # Di sini kita set untuk berjalan selama sisa epoch yang diinginkan.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=config.num_epochs, 
        eta_min=config.eta_min
    )

    # LOAD SCHEDULER STATE (PENTING AGAR SCHEDULE TIDAK RESET)
    # Cek apakah checkpoint punya data scheduler (checkpoint lama Anda mungkin belum punya)
    if load_optimizer and 'scheduler_state_dict' in checkpoint:
        print("Restoring scheduler state...")
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # ... (Dataloader Setup) ...
    train_loader, val_loader = build_dataloaders(config)

    # LOOP UTAMA
    end_epoch = start_epoch + config.num_epochs
    
    print(f"Start Training from Epoch {start_epoch} to {end_epoch-1}")

    for epoch in range(start_epoch, end_epoch):
        epoch_start = time.time()
        
        # Ambil LR sebelum training dimulai
        current_lr = optimizer.param_groups[0]['lr']
        
        # --- TRAIN ---
        train_loss, pixel_loss, edge_loss = train_epoch(model, train_loader, criterion, optimizer, epoch, config)
        
        # --- UPDATE SCHEDULER ---
        scheduler.step()
        
        # --- VALIDATE ---
        psnr, ssim, edge_psnr = validate(model, val_loader, epoch, config)
        
        # --- SAVE CHECKPOINT ---
        save_path = os.path.join(exp_dir, f"model_ft_epoch_{epoch}_{config.loss_type}.pth")
        
        # Konversi metric
        val_psnr = psnr.item() if torch.is_tensor(psnr) else psnr
        val_ssim = ssim.item() if torch.is_tensor(ssim) else ssim
        val_edge = edge_psnr.item() if torch.is_tensor(edge_psnr) else edge_psnr
        epoch_time = time.time() - epoch_start
        torch.save({
            'epoch': epoch, 
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),

            # hyperparameter
            'learning_rate': current_lr,
            'scale':config.scale,
            'window_size':config.window_size,
            'model_size':config.model_size,
            'loss_type':config.loss_type,
            'pixel_weight':config.pixel_weight,
            'sobel_weight':config.sobel_weight, 
            'betas': config.betas,
            'eta_min': config.eta_min,
            
            # meta tambahan
            "author": "kanza az zahrawani",
            "device": config.device,
            "date": datetime.now(ZoneInfo("Asia/Jakarta")).isoformat(),
            "university": "Universitas Sriwijaya",

            "train_loss":train_loss,
            "pixel_loss": pixel_loss,
            "edge_loss": edge_loss,

            "val_psnr":float(psnr),
            "val_ssim": float(ssim),
            "val_edge_psnr": float(edge_psnr),
            "epoch_time": epoch_time, 
        }, save_path)
        
        print(f"✓ Saved: {save_path} | LR used: {current_lr:.8f}\n")

class AnimeConfig:
    train_lr_dir = '/kaggle/working/dataset_split/train/LR'
    train_hr_dir = '/kaggle/working/dataset_split/train/HR'
    val_lr_dir = '/kaggle/working/dataset_split/test/LR'
    val_hr_dir = '/kaggle/working/dataset_split/test/HR'
    output_dir = '/kaggle/working/experiments_anime'

    # model parameters
    scale = 2 
    window_size = 8
    model_size = 'large'

    if model_size == 'large':
        embed_dim = 180
        depths = [6,6,6,6]
        num_heads = [6,6,6,6]
    else : 
        embed_dim = 60
        depths = [6,6,6,6]
        num_heads = [6,6,6,6]

    upsampler = 'pixelshuffle'

    loss_type = 'adaptive'
    pixel_weight = 1.0
    sobel_weight = 0.1 # for anime range between 0.1 - 0.5

    # training parameters
    batch_size = 8
    num_epochs = 2
    learning_rate = 2e-4
    betas = (0.9, 0.999)
    eta_min = 1e-7


    # device 
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # experiment name 
    experiment_name = f'anime_waifu2x_{loss_type}_x{scale}'

    if loss_type in ['combined', 'adaptive']:
        experiment_name += f'sobel : {sobel_weight}'



main_finetuning(
    checkpoint_path="/kaggle/working/model_epoch_2_adaptive.pth", 
    load_optimizer=True, 
    new_lr=None,         # <--- Set ke None agar LR lanjut dari epoch sebelumnya
    additional_epochs=2,
    new_batch_size=8
)

## Read Metadata

In [ ]:
from pprint import pprint

def read_checkpoint_metadata(checkpoint_path):
    """
    Membaca dan menampilkan metadata dari checkpoint PyTorch
    TANPA load model atau optimizer
    """
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    # Keys yang dianggap metadata (bukan state besar)
    metadata_keys = [
        "epoch",
        "learning_rate",
        "scale",
        "window_size",
        "model_size",
        "loss_type",
        "pixel_weight",
        "sobel_weight",
        "betas",
        "eta_min",
        "author",
        "university",
        "device",
        "date",
        "mode",
        "base_checkpoint",
        "train_loss",
        "pixel_loss",
        "edge_loss",
        "val_psnr",
        "val_ssim",
        "val_edge_psnr",
        "epoch_time",
    ]

    metadata = {}
    for key in metadata_keys:
        if key in checkpoint:
            metadata[key] = checkpoint[key]

    print("=" * 60)
    print("📦 CHECKPOINT METADATA")
    print("=" * 60)
    pprint(metadata)
    print("=" * 60)

    return metadata

# metadata = read_checkpoint_metadata(
#     "/kaggle/working/model_ft_epoch_7_adaptive.pth"
# )

# 

In [ ]:
metadata = read_checkpoint_metadata(
    "/kaggle/working/model_epoch_2_adaptive.pth"
)